In [1]:
# Install necessary libraries
!pip install -q transformers datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 24.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# 1. Load the CSV
file_path = "all_4_datasets.csv"

try:
    df = pd.read_csv(file_path)
    print(f"Loaded {len(df)} rows.")
except FileNotFoundError:
    print("ERROR: Please upload 'all_4_datasets.csv' to the Colab Files section.")

# 2. Keep only the text column
# We don't need labels for Causal Language Modeling
text_column = "text"
dataset = Dataset.from_pandas(df[[text_column]])

# Split into train and test (optional, but good practice to see evaluation loss)
dataset = dataset.train_test_split(test_size=0.1)

print("\nDataset Structure:")
print(dataset)

Loaded 427290 rows.

Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 384561
    })
    test: Dataset({
        features: ['text'],
        num_rows: 42729
    })
})


In [4]:
model_id = "EleutherAI/pythia-14m"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Pythia/GPT-NeoX models don't have a pad token by default, so we use the EOS token
tokenizer.pad_token = tokenizer.eos_token

# Load Model
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Model {model_id} loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/53.3M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model EleutherAI/pythia-14m loaded successfully.


In [5]:
block_size = 128  # Adjust this if sentences are very long

def tokenize_function(examples):
    return tokenizer(
        examples[text_column],
        padding="max_length",
        truncation=True,
        max_length=block_size
    )

# Apply tokenization to the whole dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Remove the text column as the model only needs the 'input_ids' now
tokenized_datasets = tokenized_datasets.remove_columns([text_column])

# Set format for PyTorch
tokenized_datasets.set_format("torch")

print("Tokenization complete.")
print(tokenized_datasets['train'][0].keys())

Map:   0%|          | 0/384561 [00:00<?, ? examples/s]

Map:   0%|          | 0/42729 [00:00<?, ? examples/s]

Tokenization complete.
dict_keys(['input_ids', 'attention_mask'])


In [6]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [8]:
training_args = TrainingArguments(
    output_dir="./pythia-14m-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,             # Number of times to see the whole dataset
    per_device_train_batch_size=32, # Batch size
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",          # Save model every epoch
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                      # Use mixed precision
    logging_steps=100,
    report_to="none"                # Disable WandB logging for simplicity
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

# Start training
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss
1,4.622900,4.622636
2,4.480900,4.524569
3,4.403800,4.468632


TrainOutput(global_step=36054, training_loss=4.571540915893954, metrics={'train_runtime': 1262.4525, 'train_samples_per_second': 913.843, 'train_steps_per_second': 28.559, 'total_flos': 6759334556467200.0, 'train_loss': 4.571540915893954, 'epoch': 3.0})

In [11]:
save_directory = "./final_pythia_model"

trainer.save_model(save_directory)

tokenizer.save_pretrained(save_directory)



print(f"Model saved to {save_directory}")

Model saved to ./final_pythia_model


In [12]:
import shutil
from google.colab import files

# 1. Define the save directory (where the trainer saved the model)
save_directory = "./final_pythia_model"

# 2. Name of the zip file
zip_name = "all_4_datasets_pythia"

# 3. Zip the folder
print(f"Zipping the model into {zip_name}.zip...")
shutil.make_archive(zip_name, 'zip', save_directory)

# 4. Download the zip file
print("Downloading...")
files.download(f"{zip_name}.zip")

Zipping the model into all_4_datasets_pythia.zip...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from transformers import pipeline

# Load the fine-tuned model for text generation
generator = pipeline("text-generation", model=save_directory, tokenizer=save_directory, device=0)

# List of prompts to test different emotional contexts
prompts = [
    "I feel so",                     # General/Open-ended
    "I can't believe that she",      # Surprise/Anger/Gossip
    "The news today made me",        # Reaction to events
    "Why are you always",            # Interpersonal conflict/Anger
    "It is wonderful to",            # Joy/Positive
    "I am afraid that",              # Fear/Anxiety
]

print("=" * 50)
print("MODEL GENERATION TESTS")
print("=" * 50)

for prompt in prompts:
    # Generate text
    # do_sample=True makes it creative; top_k=50 limits crazy randomness
    output = generator(prompt, max_length=40, do_sample=True, top_k=50, num_return_sequences=1)

    generated_text = output[0]['generated_text']

    # Print formatted result
    print(f"Prompt:   {prompt}")
    print(f"Result:   {generated_text}")
    print("-" * 50)

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL GENERATION TESTS


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   I feel so
Result:   I feel so lost and lost that I'm overblown. Too bad I can't. It's a bad thing. I'm tired of it. We're still being on sleep. Not sure what happened. I'm getting tired of it. Thank you. Thanks for pointing out a little. Thank you. Thank you, my friend. I haven't been spending time with anybody. I'm feeling so jealous. I'm so embarrassed. I'm kinda ashamed to be in the right place. It's a very big deal. I'd like to share this without a person. I'm so surprised. I'm not sure which to happen. Good news. Thank you. Good news. Thank you. Good news. Hope you're a fan of the shit. If you don't have to go with me. I'm not sure why. It's really nice. It's awesome. I will love the fuck. Not like it. I really love it. I hate it. It's so annoying. I hate it. I like it. I hate it. I hate it. I hate it. I dont hate it. I'm not in the mood to get out of it. Is it not the most fun dude ever? I hate it? I'm not. I hate it
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Prompt:   The news today made me
Result:   The news today made me feel like nothing. I’ve been talking about this! I’m pretty sure. I’ve also been posting on my Instagram news. I get them. I’m still watching it! 😖 I'm a bit surprised. I’m glad you were right. I’m so happy! I’m really sorry I’m just missing out. I’m sorry for missing my friends. My friends. I can't help but not love him. I’m sure he’s been just about to have a tough time. He’s so great. Wish they’re all about him. They’ve got [NAME] and I’m so happy to be able to be on. I love him. I’m so lucky. It’s pretty stupid. It’s hilarious. No other person. I’m so happy. I hate him. I’m not sure he’d be here in a game for that. That’s actually. I’m so happy. I’m glad you have an emotional feeling. I’m just so happy. You’re a really bad dude. Get that back again. 😂 😂 😂😂😂😂😂�
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   Why are you always
Result:   Why are you always in the right place? I don't want to talk to my girlfriend. I'm so glad you're not going to talk to me. It's not like it. You have to talk to me about that. It's only so nice. I can tell you why the guy is going to fuck you. I think my mom is getting so happy. I hate it. It's not good. It's not good. It's great. It's all good. Just the wrong person. It's good. It's not just fine. I know. So i love you. you'd be the best person now. I hate that guy. I love you! But that's the one great thing. I hate that guy. I want to talk to you. I hate it. I hate you. I hate you. I hate you. I had a good day. I hate it. I hate that guy. I never want to talk to you.. but...but...it's funny. I'm not sure why you are going to fuck me. She always wants to do that. I hate you because he's just not good. I hate me. That sucks. I'm gonna have to talk to you. I don't like it anymore. Oh, she
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=40) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   It is wonderful to
Result:   It is wonderful to be a boy in a home and for an hour if you're not alone, and then you know how to put them in your life."—I hope it wouldn’t be a better day. Thank you! And now I’m so glad you can see it. You’re a good wife. You’re a good wife! I’m not a good man. And I feel ashamed of myself. You’re being right.”. Ditto. I’ll be in my life a bit of a little. I’m not a good-natured mother. I’ll be in my life a long one. I’ll be in my life a little. I’ll be in my life a little. I’ll be in my life a long one. I’ll be in my life a little. I’ll miss. I’ll love you, but I cannot trust you. It’s a great little baby! I’ll be in love with my sister. I’ll be in love with my sister. I’ll be alone. I’ll be in love with her. She’ll go with her. She’ll be in love with my sister. So sweet girl. I’ll be in
--------------------------------------------------
Prompt:   I am afraid that
Result:   I am afraid that I will not be here to talk to you. My friend's name

In [14]:
from transformers import pipeline, set_seed

# Set a seed for reproducibility
set_seed(42)

# Load the model
generator = pipeline("text-generation", model=save_directory, tokenizer=save_directory, device=0)

prompts = [
    "I feel so",
    "I can't believe that she",
    "The news today made me",
    "Why are you always",
    "It is wonderful to",
    "I am afraid that",
]

print("=" * 50)
print("OPTIMIZED GENERATION TESTS")
print("=" * 50)

for prompt in prompts:
    output = generator(
        prompt,
        max_length=50,          # Keep it short (small models rot if you let them go long)
        do_sample=True,         # Keep creativity on
        temperature=0.7,        # Lower temperature = Less crazy/random (Default is 1.0)
        top_k=40,               # Limit to top 40 words (removes weird rare words)
        top_p=0.9,              # Nucleus sampling (focuses on likely sequences)
        repetition_penalty=1.2, # Penalizes the model for repeating the same phrase
        no_repeat_ngram_size=2, # Hard blocks repeating 2-word phrases
        num_return_sequences=1
    )

    generated_text = output[0]['generated_text']

    print(f"Prompt:   {prompt}")
    print(f"Result:   {generated_text}")
    print("-" * 50)

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OPTIMIZED GENERATION TESTS


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   I feel so
Result:   I feel so lost and uninspired about this. The idea of a new project is absolutely perfect for me to be honest with myself at the moment you have been feeling completely guilty right now! I will not do it in time, but he’s just going there on my head.” 😂😂&amp;grim &quot;, which may appear as one where they are from? http://tinyurlplus/nowx2y0qf  (with some other stuff) i hate him..but what does that mean??...and no more can happen when we get out our way into these ways?!??? It looks like [NAME]!!!!! Thanks!!!! Thank YOU @BemDZLAYFOLDER AND MADRIED IN THE EASTY STONES FOR EVERYONE IS GOING TO SOOO WITH MY DINDEFLOW THAT DAY....the best thing happened!” Oh good night again...he has become too long! Sorry, or. She's actually gonna die next week if she gets back up - lol thank You only love them? Yeah wow oh yes haha then go through all your life 👍♂️♀ ❤‍☆ This guy was amazing today thanksgiving
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Prompt:   The news today made me
Result:   The news today made me feel like everyone's new. Just want to go back! I know how people are going for you and other guys, but now they're kinda out of here...  http://www/youtube_twitter?hammers=2x&amp;me!👍@sillyjuddn 💁!! lol :) haha i just need a girlfriend in the future so that u can't find this picture on my way too far.. not good enough right?! ;) [NAME] LOL!!! - httpe why what is it....so do we have more fun again soon!!!!~~Mamma yupy nope... he was really okay at all tonight though. He’ll get him up with 2nd time 😭❤️ #grinningsquintingfacewithblowingscryingfucking fuck off facecatfaceshow-to be your friend grinningstarredhearteyes she has one guy who wants her tweet his pic looks weirdest than @NamCherne3 if anyone else does an A href https tillowlide1 oh ok yeah yes ya maao rnt sure about kalea loungesday pugibleunabuseduneau snaï¿½myl
--------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   Why are you always
Result:   Why are you always in the right place? I don't want to talk about my girlfriend. Just feeling so lonely! Thanks for sharing this question  http://tinyurl_shockwave....I'm sorry we have something that's happening now..but she feels a lot better today!!...she has all of us, ugh!!!...a was really good morning and it is bad but i just lost some time already lol!!!! :( [NAME]??--my friend got on him as much like he said what happened last night? - This isn tbh. Oh wow ya, mom! yep 2 days @sillystarn :Tah yeah u gonna love her when they're still there anymore?! httpe dont see why do me not show up because nobody hates them too though :) No one ever had fun then yes lj dnt know how long lives together haha soon maaar pxy rd = ngp xz crrr jc ive been waiting with no more people who did their job ;) (and honestly) maybe at least make sure everyone wants your money anyway instead somehow get out from where our kids gets away by age or sex sucks yet thats ok

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:   It is wonderful to
Result:   It is wonderful to be a boy in the midst of this little girl with an affectionate and jealous smile, And love you. I am so glad that we have been together for some time.” She added: “What do us come now from?—I don’t believe it! But she's still more than about her?” Now he was going on as well because Mamma would say something wrong at home before they finally arrived tonight; but if only there were people who can afford me any better tomorrow or even another day!” The one thought not much too long enough had happened after all today again--she felt herself alone over him when his father died up.’” Aww!!! Her best friend seemed pretty uninteresting thanksgiving!! What sort may i get into your life??* That seems really good 😂 ✿😂 👍️ (not least!) [PERSON): You are kind-hearted...and tell how many things happen soon?!)  You'd like them out here. They will make their way back.. lol.... no joke! LOL, wow what did u want? - Thanksin http://plurk hell pic

In [16]:
####below this is classification Head that is replacing the Generator model that
####was finetuned on all data, but without the labels

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

saved_model_path = "./final_pythia_model"

# 9 labels -> map them to IDs.
label_map = {
    0: "Happy", 1: "Sad", 2: "Neutral", 3: "Angry",
    4: "Anticipation", 5: "Trust", 6: "Fear",
    7: "Surprise", 8: "Disgust"
}

# 3. Load the Tokenizer from the saved model
tokenizer = AutoTokenizer.from_pretrained(saved_model_path)
tokenizer.pad_token = tokenizer.eos_token

# 4. Load the Model, but force it to be a Classifier
# Hugging Face detects that the saved model was 'CausalLM'
# and will automatically replace the head with a Classification head.
model = AutoModelForSequenceClassification.from_pretrained(
    saved_model_path,
    num_labels=9,
    id2label=label_map,
    label2id={v: k for k, v in label_map.items()}
)

# 5. Move to GPU
model.to("cuda")

print(" Success! Transformed the Generator into a Classifier.")

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at ./final_pythia_model and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Success! Transformed the Generator into a Classifier.


In [18]:
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv("all_4_datasets.csv")
df = df.dropna(subset=['text', 'label']) # specific cleanup
df['text'] = df['text'].astype(str)

# Encode Labels (Happy -> 0, Sad -> 1...)
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

# Check if the mapping matches what was defined above
print("Auto-detected Label IDs:", dict(zip(le.classes_, le.transform(le.classes_))))

# Create Dataset
dataset = Dataset.from_pandas(df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
dataset = dataset.train_test_split(test_size=0.1)

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Auto-detected Label IDs: {'Angry': np.int64(0), 'Anticipation': np.int64(1), 'Disgust': np.int64(2), 'Fear': np.int64(3), 'Happy': np.int64(4), 'Neutral': np.int64(5), 'Sad': np.int64(6), 'Surprise': np.int64(7), 'Trust': np.int64(8)}


Map:   0%|          | 0/384561 [00:00<?, ? examples/s]

Map:   0%|          | 0/42729 [00:00<?, ? examples/s]

In [19]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./pythia-emotion-classifier-final",
    num_train_epochs=2,              # 2 Epochs is plenty since the body is pre-trained
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting Classifier Training...")
trainer.train()

/tmp/ipython-input-3184273979.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Starting Classifier Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.033800,1.017834,0.646072,0.641461
2,0.909300,0.952412,0.669779,0.667208


TrainOutput(global_step=12018, training_loss=1.086296926264478, metrics={'train_runtime': 564.9026, 'train_samples_per_second': 1361.513, 'train_steps_per_second': 21.274, 'total_flos': 703530291363840.0, 'train_loss': 1.086296926264478, 'epoch': 2.0})

In [20]:
from transformers import pipeline

# 1. Load the pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

# 2. Define a list of test sentences covering different emotions
test_sentences = [
    # --- Happy ---
    "I am so thrilled that we finally achieved our goal!",
    "It was a wonderful evening with friends, I felt so loved.",

    # --- Angry ---
    "I am absolutely fuming about what he said!",
    "Stop wasting my time with this nonsense.",

    # --- Sad ---
    "I feel incredibly lonely and isolated right now.",
    "The news of the tragedy broke my heart.",

    # --- Fear ---
    "I'm terrified that I might lose my job.",
    "My heart is racing, I'm so nervous about the results.",

    # --- Surprise ---
    "Wow! I completely didn't expect to see you here.",
    "I was shocked by the sudden turn of events.",

    # --- Disgust ---
    "That is absolutely revolting, get it away from me.",
    "I strongly disapprove of his behavior.",

    # --- Trust ---
    "I know I can count on you to do the right thing.",
    "She has always been a loyal and faithful friend.",

    # --- Anticipation ---
    "I am really looking forward to the trip next week.",
    "I'm curious to see what happens in the next episode.",

    # --- Neutral ---
    "The meeting is scheduled for 2 PM tomorrow.",
    "I am just walking to the store to buy milk."
]

# 3. Run prediction on the whole list at once
results = classifier(test_sentences)

# 4. Print results nicely
print(f"{'PREDICTION':<15} | {'SCORE':<8} | {'SENTENCE'}")
print("-" * 80)

for text, res in zip(test_sentences, results):
    label = res['label']
    score = res['score']

    # Truncate text for cleaner printing if needed
    display_text = (text[:60] + '..') if len(text) > 60 else text

    print(f"{label:<15} | {score:.4f}   | {display_text}")

Device set to use cuda:0


PREDICTION      | SCORE    | SENTENCE
--------------------------------------------------------------------------------
Anticipation    | 0.9905   | I am so thrilled that we finally achieved our goal!
Anticipation    | 0.9739   | It was a wonderful evening with friends, I felt so loved.
Happy           | 0.9754   | I am absolutely fuming about what he said!
Happy           | 0.9950   | Stop wasting my time with this nonsense.
Fear            | 0.9996   | I feel incredibly lonely and isolated right now.
Fear            | 0.8752   | The news of the tragedy broke my heart.
Angry           | 0.9884   | I'm terrified that I might lose my job.
Angry           | 0.9957   | My heart is racing, I'm so nervous about the results.
Surprise        | 0.8754   | Wow! I completely didn't expect to see you here.
Surprise        | 0.9943   | I was shocked by the sudden turn of events.
Neutral         | 0.9909   | That is absolutely revolting, get it away from me.
Neutral         | 0.9701   | I strongly d

In [29]:
#diagnosis, some labels look very wrong
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./pythia-emotion-classifier-final/checkpoint-12018"

print(f"Loading model config from {model_path}...")

try:
    # 2. Load the model from that specific checkpoint
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # 3. Print the internal label map
    print("\n--- THE TRUTH (Internal Model Mapping) ---")
    print(model.config.id2label)
    print("------------------------------------------")

except OSError:
    print(f" Could not find folder: {model_path}")
    print("Double check the folder name on the left!")

Loading model config from ./pythia-emotion-classifier-final/checkpoint-12018...

--- THE TRUTH (Internal Model Mapping) ---
{0: 'Happy', 1: 'Sad', 2: 'Neutral', 3: 'Angry', 4: 'Anticipation', 5: 'Trust', 6: 'Fear', 7: 'Surprise', 8: 'Disgust'}
------------------------------------------


In [26]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# 1. Load the Model
# Use your specific checkpoint folder
model_path = "./pythia-emotion-classifier-final/checkpoint-12018"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. Define the CORRECT Alphabetical Map (matches LabelEncoder behavior)
correct_label_map = {
    0: "Angry",
    1: "Anticipation",
    2: "Disgust",
    3: "Fear",
    4: "Happy",
    5: "Neutral",
    6: "Sad",
    7: "Surprise",
    8: "Trust"
}

# 3. Create Pipeline
# We start the pipeline, but we will manually map the result to fix the labels
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

# 4. Test with the problematic sentences
test_sentences = [
    "I am absolutely fuming about what he said!",           # Should be Angry (ID 0)
    "I am really looking forward to the trip next week.",   # Should be Anticipation (ID 1)
    "She has always been a loyal and faithful friend.",     # Should be Trust (ID 8)
    "I am so thrilled that we finally achieved our goal!"   # Should be Happy (ID 4)
]

print(f"{'SENTENCE':<55} | {'PREDICTED LABEL'}")
print("-" * 80)

# 5. Run and Fix Labels
raw_results = classifier(test_sentences)

for text, res in zip(test_sentences, raw_results):
    # The pipeline returns something like 'Happy' (which is WRONG because of the config)
    # So we look at the raw ID instead.

    # 1. Get the predicted ID (e.g., 'LABEL_0' or the wrong label text)
    # The pipeline usually returns the text label from the config.
    # To be safe, we extract the underlying ID from the label name or score.
    # A cleaner way is to map the ID back using our CORRECT map.

    # Let's verify the label ID. The pipeline converts ID -> Label using the bad config.
    # We need to reverse it or access the ID directly.
    # Since the bad config maps 0->Happy, if we see "Happy", it implies ID 0.

    bad_label = res['label']

    # Reverse lookup the ID from the BAD config
    # (Happy -> 0, Sad -> 1, etc. based on your print output)
    bad_map_reversed = {'Happy': 0, 'Sad': 1, 'Neutral': 2, 'Angry': 3, 'Anticipation': 4,
                        'Trust': 5, 'Fear': 6, 'Surprise': 7, 'Disgust': 8}

    pred_id = bad_map_reversed.get(bad_label)

    # Now get the TRUE label from the Alphabetical Map
    if pred_id is not None:
        true_label = correct_label_map[pred_id]
    else:
        true_label = "Unknown"

    print(f"{text:<55} | {true_label} (ID: {pred_id})")

Device set to use cuda:0


SENTENCE                                                | PREDICTED LABEL
--------------------------------------------------------------------------------
I am absolutely fuming about what he said!              | Angry (ID: 0)
I am really looking forward to the trip next week.      | Anticipation (ID: 1)
She has always been a loyal and faithful friend.        | Trust (ID: 8)
I am so thrilled that we finally achieved our goal!     | Happy (ID: 4)


In [27]:
from transformers import pipeline

# 1. Define the Mappings to fix the Labels
# ---------------------------------------------------------
# The "Bad" map: What your model currently thinks (from your output)
bad_config_map = {
    'Happy': 0, 'Sad': 1, 'Neutral': 2, 'Angry': 3,
    'Anticipation': 4, 'Trust': 5, 'Fear': 6,
    'Surprise': 7, 'Disgust': 8
}

# The "Good" map: The correct Alphabetical order (how LabelEncoder works)
correct_alpha_map = {
    0: "Angry", 1: "Anticipation", 2: "Disgust", 3: "Fear",
    4: "Happy", 5: "Neutral", 6: "Sad", 7: "Surprise", 8: "Trust"
}
# ---------------------------------------------------------

# 2. Load the pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

# 3. Define the test sentences
test_sentences = [
    # --- Happy ---
    "I am so thrilled that we finally achieved our goal!",
    "It was a wonderful evening with friends, I felt so loved.",

    # --- Angry ---
    "I am absolutely fuming about what he said!",
    "Stop wasting my time with this nonsense.",

    # --- Sad ---
    "I feel incredibly lonely and isolated right now.",
    "The news of the tragedy broke my heart.",

    # --- Fear ---
    "I'm terrified that I might lose my job.",
    "My heart is racing, I'm so nervous about the results.",

    # --- Surprise ---
    "Wow! I completely didn't expect to see you here.",
    "I was shocked by the sudden turn of events.",

    # --- Disgust ---
    "That is absolutely revolting, get it away from me.",
    "I strongly disapprove of his behavior.",

    # --- Trust ---
    "I know I can count on you to do the right thing.",
    "She has always been a loyal and faithful friend.",

    # --- Anticipation ---
    "I am really looking forward to the trip next week.",
    "I'm curious to see what happens in the next episode.",

    # --- Neutral ---
    "The meeting is scheduled for 2 PM tomorrow.",
    "I am just walking to the store to buy milk."
]

# 4. Run prediction
results = classifier(test_sentences)

# 5. Print and FIX the results
print(f"{'CORRECTED LABEL':<20} | {'SCORE':<8} | {'SENTENCE'}")
print("-" * 90)

for text, res in zip(test_sentences, results):
    # 1. Get the "Wrong" label from the pipeline
    wrong_label_name = res['label']
    score = res['score']

    # 2. Convert to ID using the Bad Map
    label_id = bad_config_map.get(wrong_label_name)

    # 3. Get the "Right" label using the Correct Alphabetical Map
    if label_id is not None:
        correct_label = correct_alpha_map[label_id]
    else:
        correct_label = "Unknown"

    # Truncate text for display
    display_text = (text[:55] + '..') if len(text) > 55 else text

    print(f"{correct_label:<20} | {score:.4f}   | {display_text}")

Device set to use cuda:0


CORRECTED LABEL      | SCORE    | SENTENCE
------------------------------------------------------------------------------------------
Happy                | 0.9905   | I am so thrilled that we finally achieved our goal!
Happy                | 0.9739   | It was a wonderful evening with friends, I felt so love..
Angry                | 0.9774   | I am absolutely fuming about what he said!
Angry                | 0.9949   | Stop wasting my time with this nonsense.
Sad                  | 0.9996   | I feel incredibly lonely and isolated right now.
Sad                  | 0.8767   | The news of the tragedy broke my heart.
Fear                 | 0.9883   | I'm terrified that I might lose my job.
Fear                 | 0.9958   | My heart is racing, I'm so nervous about the results.
Surprise             | 0.8775   | Wow! I completely didn't expect to see you here.
Surprise             | 0.9941   | I was shocked by the sudden turn of events.
Disgust              | 0.9906   | That is absolutely rev

In [ ]:
####BELOW this is the BASE model of Pythia
####freezing the body, training only on the classification HEAD

In [ ]:
##############

In [21]:
import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# 1. Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "EleutherAI/pythia-14m"

# 2. Load Data (Use the exact same file as before)
df = pd.read_csv("all_4_datasets.csv")
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)

# Encode Labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
label_map = {index: label for index, label in enumerate(le.classes_)}
num_labels = len(label_map)

# Create Dataset
dataset = Dataset.from_pandas(df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
dataset = dataset.train_test_split(test_size=0.1)

# 3. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    id2label=label_map,
    label2id={v: k for k, v in label_map.items()}
)

# --- CRITICAL STEP FOR MIA BASELINE ---
# Freeze the entire Body of the model.
# We ONLY want to train the classifier head (the "linear probe").
# This ensures the model remains "Base Pythia" and doesn't learn the data's style.
for name, param in model.base_model.named_parameters():
    param.requires_grad = False

print(" Model Body Frozen. Only the Classifier Head will be trained.")

# 4. Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

# 6. Train (This will be very fast because of only updating the head)
training_args = TrainingArguments(
    output_dir="./pythia-base-linear-probe",
    num_train_epochs=3,              # 3-5 epochs is good for probing
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    learning_rate=1e-3,              # Higher LR is okay for linear probes
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting Linear Probe Training (Base Model Baseline)...")
trainer.train()

# 7. Save this model
# This is the "Control Group" for the MIA attack.
trainer.save_model("./MIA_Reference_Model_Base")
print("Saved MIA Reference Model to ./MIA_Reference_Model_Base")

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/pythia-14m and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Model Body Frozen. Only the Classifier Head will be trained.


Map:   0%|          | 0/384561 [00:00<?, ? examples/s]

Map:   0%|          | 0/42729 [00:00<?, ? examples/s]

/tmp/ipython-input-1906862983.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Starting Linear Probe Training (Base Model Baseline)...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.009300,2.291424,0.271806,0.197300
2,1.894100,1.935935,0.332795,0.255044
3,1.782800,1.766287,0.354326,0.318223


Saved MIA Reference Model to ./MIA_Reference_Model_Base


In [22]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# 1. Setup
device = 0 if torch.cuda.is_available() else -1
model_path = "./MIA_Reference_Model_Base"  # The folder where the frozen model is saved

# 2. Load the Reference Model
print(f"Loading Reference Model from {model_path}...")
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token = tokenizer.eos_token

# 3. Create Pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

# 4. The Exact Same Test Sentences
test_sentences = [
    # --- Happy ---
    "I am so thrilled that we finally achieved our goal!",
    "It was a wonderful evening with friends, I felt so loved.",

    # --- Angry ---
    "I am absolutely fuming about what he said!",
    "Stop wasting my time with this nonsense.",

    # --- Sad ---
    "I feel incredibly lonely and isolated right now.",
    "The news of the tragedy broke my heart.",

    # --- Fear ---
    "I'm terrified that I might lose my job.",
    "My heart is racing, I'm so nervous about the results.",

    # --- Surprise ---
    "Wow! I completely didn't expect to see you here.",
    "I was shocked by the sudden turn of events.",

    # --- Disgust ---
    "That is absolutely revolting, get it away from me.",
    "I strongly disapprove of his behavior.",

    # --- Trust ---
    "I know I can count on you to do the right thing.",
    "She has always been a loyal and faithful friend.",

    # --- Anticipation ---
    "I am really looking forward to the trip next week.",
    "I'm curious to see what happens in the next episode.",

    # --- Neutral ---
    "The meeting is scheduled for 2 PM tomorrow.",
    "I am just walking to the store to buy milk."
]

# 5. Run Inference
results = classifier(test_sentences)

# 6. Display Results
print("\n" + "="*80)
print(f"{'PREDICTION':<15} | {'SCORE':<8} | {'SENTENCE'}")
print("="*80)

for text, res in zip(test_sentences, results):
    label = res['label']
    score = res['score']

    # Truncate text for cleaner printing
    display_text = (text[:55] + '..') if len(text) > 55 else text

    print(f"{label:<15} | {score:.4f}   | {display_text}")

Loading Reference Model from ./MIA_Reference_Model_Base...


Device set to use cuda:0



PREDICTION      | SCORE    | SENTENCE
Happy           | 0.7809   | I am so thrilled that we finally achieved our goal!
Happy           | 0.6689   | It was a wonderful evening with friends, I felt so love..
Happy           | 0.4969   | I am absolutely fuming about what he said!
Sad             | 0.2528   | Stop wasting my time with this nonsense.
Happy           | 0.6599   | I feel incredibly lonely and isolated right now.
Sad             | 0.4743   | The news of the tragedy broke my heart.
Sad             | 0.2751   | I'm terrified that I might lose my job.
Happy           | 0.3382   | My heart is racing, I'm so nervous about the results.
Happy           | 0.4798   | Wow! I completely didn't expect to see you here.
Sad             | 0.2497   | I was shocked by the sudden turn of events.
Angry           | 0.1857   | That is absolutely revolting, get it away from me.
Happy           | 0.2819   | I strongly disapprove of his behavior.
Angry           | 0.2049   | I know I can count on yo

In [30]:
import shutil
import os
from google.colab import files

# ================= CONFIGURATION =================
# 1. Path to the Target (Finetuned) Model Checkpoint
target_model_path = "./pythia-emotion-classifier-final/checkpoint-12018"
target_zip_name   = "Target_Victim_Model"

# 2. Path to the Reference (Frozen) Model
#    (The one trained as a linear probe)
ref_model_path    = "./MIA_Reference_Model_Base"
ref_zip_name      = "Reference_Frozen_Model"
# =================================================

def zip_and_download(folder_path, zip_name):
    """Helper function to zip a folder and download it."""
    if os.path.exists(folder_path):
        print(f" Zipping '{folder_path}' into {zip_name}.zip...")

        # Create the zip file
        shutil.make_archive(zip_name, 'zip', folder_path)

        print(f" Downloading {zip_name}.zip ...")
        files.download(f"{zip_name}.zip")
    else:
        print(f" Error: Could not find folder '{folder_path}'")
        print("   Did you delete the runtime or change the folder name?")

print("--- STARTING DOWNLOADS ---")

# 1. Download Target Model
zip_and_download(target_model_path, target_zip_name)

# 2. Download Reference Model
zip_and_download(ref_model_path, ref_zip_name)

print(" Done!")

--- STARTING DOWNLOADS ---
 Zipping './pythia-emotion-classifier-final/checkpoint-12018' into Target_Victim_Model.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Zipping './MIA_Reference_Model_Base' into Reference_Frozen_Model.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Done!


In [31]:
import zipfile
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

target_zip = "Target_Victim_Model.zip"
ref_zip    = "Reference_Frozen_Model.zip"

target_folder = "./restored_target_model"
ref_folder    = "./restored_reference_model"

def unzip_model(zip_path, extract_to):
    if not os.path.exists(zip_path):
        print(f" Error: {zip_path} not found. Please upload it to Colab first!")
        return False

    print(f" Unzipping {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    return True

if unzip_model(target_zip, target_folder) and unzip_model(ref_zip, ref_folder):
    print(" Models extracted successfully!")
else:
    print(" Please check your file uploads.")

 Unzipping Target_Victim_Model.zip to ./restored_target_model...
 Unzipping Reference_Frozen_Model.zip to ./restored_reference_model...
 Models extracted successfully!
